# LSTM Next-Word Suggestions

Load the saved LSTM and the exact vocabulary/configuration produced by the PDF training pipeline.

In [1]:
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import tensorflow as tf

PROJECT_ROOT = Path.cwd().parent
MODEL_PATH = PROJECT_ROOT / 'models' / 'technical_writing_lstm_best.keras'
VOCABULARY_PATH = PROJECT_ROOT / 'data' / 'processed' / 'vocabulary.json'
CONFIG_PATH = PROJECT_ROOT / 'data' / 'processed' / 'model_config.json'

model = tf.keras.models.load_model(MODEL_PATH)
vocabulary_data = json.loads(VOCABULARY_PATH.read_text(encoding='utf-8'))
model_config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
word_to_index = vocabulary_data['word_to_index']
index_to_word = {int(index): word for index, word in vocabulary_data['index_to_word'].items()}
SEQUENCE_LENGTH = model_config['sequence_length']
SPECIAL_TOKENS = {'<PAD>', '<UNK>', '<START>', '<END>'}
TOKEN_PATTERN = re.compile(model_config['token_pattern'])

def preprocess_text(text):
    return TOKEN_PATTERN.findall(unicodedata.normalize('NFKC', text).lower().replace('\u00ad', ''))

print(f'Loaded LSTM with vocabulary size {len(word_to_index):,}.')

Loaded LSTM with vocabulary size 3,282.


## Context-aware next-word function

Training used a `<START>` token followed by the words typed so far, left-padded to 20 positions. This function recreates that representation and filters special tokens only from the displayed suggestions.

In [2]:
def get_next_word_suggestions(text, number_of_suggestions=3):
    if not text or not text.strip():
        return []
    tokens = preprocess_text(text)
    token_ids = [word_to_index['<START>']] + [word_to_index.get(token, word_to_index['<UNK>']) for token in tokens]
    token_ids = token_ids[-SEQUENCE_LENGTH:]
    padded = [word_to_index['<PAD>']] * (SEQUENCE_LENGTH - len(token_ids)) + token_ids
    probabilities = model.predict(np.asarray([padded], dtype=np.int32), verbose=0)[0]
    suggestions = []
    for index in np.argsort(probabilities)[::-1]:
        word = index_to_word[int(index)]
        if word not in SPECIAL_TOKENS:
            suggestions.append({'word': word, 'probability': float(probabilities[index])})
        if len(suggestions) == number_of_suggestions:
            break
    return suggestions

## Try a prediction

Enter text drawn from the Technical Writing corpus. The LSTM returns the most likely immediate next words.

In [ ]:
print("Type text and press Enter for suggestions. Type 'exit' and press Enter to stop.")
while True:
    user_text = input('\nEnter your text: ').strip()
    if user_text.lower() == 'exit':
        print('Prediction session ended.')
        break
    suggestions = get_next_word_suggestions(user_text)
    if not suggestions:
        print('Please enter at least one word.')
        continue
    print(f'Input: {user_text}')
    for rank, suggestion in enumerate(suggestions, start=1):
        print(f"{rank}. {suggestion['word']:<20} {suggestion['probability'] * 100:.2f}%")